# FABN Optimizer — Data Pipeline

Builds every input array the optimizer needs from the three BigQuery tables:
- `Asset_Cashflows` — cashflow schedule per CUSIP
- `Agg_Spread_Long` — daily spread per CUSIP
- `Agg_Fixed_Field` — static bond attributes (rating, duration, sector, …)

**Outputs** (all stored in `pipeline` dict at the bottom):

| Key | Shape | Description |
|---|---|---|
| `CUSIPS` | (N,) | ordered bond universe |
| `spread` | (N,) | OAS spread in decimal (bps / 10 000) |
| `durs` | (N,) | modified duration (years) |
| `theta` | (N,) | C-1 RBC charge factor |
| `h_curr` | (N,) | current equal-weight allocation |
| `bond_cf` | (T, N) | daily cashflow matrix |
| `qtr_bond_cf` | (Q, N) | quarterly cashflow matrix |
| `qtr_idx` | (Q,) | quarter labels |
| `t_vec` | (T,) | time in years from optimization date |

Deferred (set to zeros for now, plug in later):
- `tau` — transaction costs
- `signal` — composite signal
- `score` — recomputed once signal is ready

---
## 0 — Imports & Connection

In [1]:
from google.cloud import bigquery
import numpy as np
import pandas as pd
import warnings

# Scope warning suppression to the known-noisy deprecation chatter from the
# BigQuery / pandas stack, rather than blanket-silencing *everything*. Real
# RuntimeWarnings (e.g. divide-by-zero in the math) are still surfaced.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Shared financial math (single source of truth, unit-tested in tests/).
import fabn_finance_SWAP as ff

PROJECT_ID = "insurance-backed-securities"
DATASET    = "Securities"

client = bigquery.Client(project=PROJECT_ID)
print(f"Connected: {client.project}")

Connected: insurance-backed-securities


---
## 1 — Parameters

Set the optimization date and the FABN liability parameters here.
Everything downstream is derived from `optimization_date`.

In [2]:
import datetime

# ── Optimization date ──────────────────────────────────────────────────────
optimization_date = pd.Timestamp("2024-03-29")

# ── FABN terms ─────────────────────────────────────────────────────────────
FABN_ISSUE    = pd.Timestamp("2022-09-06")
FABN_MATURITY = pd.Timestamp("2027-09-06")
FABN_COUPON   = 0.03205   # 3.205% annual, paid semi-annually

# ── FABN / liability parameters (Athene-sourced) ──────────────────────────
H       = 500_000_000.0   # total capital budget ($) — CONFIRM WITH ATHENE
r_FABN  = FABN_COUPON     # funding agreement crediting rate (annual, decimal)
# D_FABN is computed in Section 8 from the actual cashflow schedule
C_curr  = 5_000_000.0     # current regulatory capital ($) — CONFIRM WITH ATHENE
C_min   = 1_000_000.0     # minimum required capital ($)   — CONFIRM WITH ATHENE
RBC_bar = 3.0             # minimum RBC solvency ratio (base case, per paper Appendix A)
dt      = 1.0             # time scaling factor (1 = annual)

# ── Optimizer penalty weights (tune later) ────────────────────────────────
gamma_w  = 0.15  # γ : weight on capital cost (C1 + C3)
beta_w   = 0.0   # β : weight on signal  (0 until signal is ready)
alpha_w  = 0.0   # α : C3 duration mismatch scaling (0 until C3 is active)
lambda_w = 0.05   # λ : CF shortfall penalty weight
eps_D    = 0.3   # duration tolerance band (years)

print(f"Optimization date  : {optimization_date.date()}")
print(f"FABN issue/maturity: {FABN_ISSUE.date()} → {FABN_MATURITY.date()}")
print(f"Budget H           : ${H:,.0f}")
print(f"r_FABN             : {r_FABN*100:.3f}%")

Optimization date  : 2024-03-29
FABN issue/maturity: 2022-09-06 → 2027-09-06
Budget H           : $500,000,000
r_FABN             : 3.205%


In [3]:
gamma_w

0.15

---
## 2 — Bond Universe from `Agg_Fixed_Field`

Defines the bond index `i = 0 … N-1`.
Every downstream array is aligned to this CUSIP list.

In [4]:
sql_fixed = f"""
SELECT
    CUSIP,
    `Amt Out`          AS amt_out,
    Cpn                AS coupon,
    Maturity           AS maturity,
    `BBG Composite`    AS rating_sp,
    `Mac Dur _Ask_`    AS mac_dur_bbg,
    RTG_MOODY          AS rating_moodys,
    BICS_LEVEL_1_SECTOR_NAME  AS sector,
    CPN_FREQ           AS cpn_freq
FROM `{PROJECT_ID}.{DATASET}.Agg_Fixed_Field`
WHERE CUSIP IS NOT NULL
  AND Maturity > '{optimization_date.date()}'
"""

fixed = client.query(sql_fixed).to_dataframe()
fixed["maturity"] = pd.to_datetime(fixed["maturity"])

# Deduplicate — keep one row per CUSIP
fixed = fixed.drop_duplicates(subset="CUSIP").reset_index(drop=True)

CUSIPS = fixed["CUSIP"].tolist()
N      = len(CUSIPS)
cusip_idx = {c: i for i, c in enumerate(CUSIPS)}   # fast reverse lookup

print(f"Universe size N = {N} bonds")
fixed.head()

c:\Users\lalot\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Universe size N = 303 bonds


,CUSIP,amt_out,coupon,maturity,rating_sp,mac_dur_bbg,rating_moodys,sector,cpn_freq
0,58769JAQ0,800000000,4.80,2027-01-11,A,0.835555,A2,Consumer Discretionary,2
1,YT2012888,600000000,5.10,2029-11-15,A,3.366354,A2,Consumer Discretionary,2
2,05565ECH6,650000000,4.90,2027-04-02,A,1.036919,A2,Consumer Discretionary,2
3,233835AQ0,1500000000,8.50,2031-01-18,A,4.119051,A2,Consumer Discretionary,2
4,58769JAC1,500000000,5.25,2027-11-29,A,1.656148,A2,Consumer Discretionary,2


---
## 3 — Spreads from `Agg_Spread_Long`

Pull the spread for each CUSIP on (or nearest to) `optimization_date`.
Spread is stored in **basis points** — we convert to decimal for the optimizer.

In [5]:
sql_spread = f"""
WITH ranked AS (
    SELECT
        CUSIP,
        Spread,
        Date,
        ROW_NUMBER() OVER (
            PARTITION BY CUSIP
            ORDER BY ABS(DATE_DIFF(Date, DATE '{optimization_date.date()}', DAY))
        ) AS rn
    FROM `{PROJECT_ID}.{DATASET}.Agg_Spread_Long`
    WHERE Date BETWEEN DATE_SUB(DATE '{optimization_date.date()}', INTERVAL 5 DAY)
                   AND DATE_ADD(DATE '{optimization_date.date()}', INTERVAL 5 DAY)
)
SELECT CUSIP, Spread, Date
FROM ranked
WHERE rn = 1
"""

spread_df = client.query(sql_spread).to_dataframe()
spread_map = spread_df.set_index("CUSIP")["Spread"]

# Align to CUSIP order; convert bps → decimal
spread_bps = np.array([spread_map.get(c, np.nan) for c in CUSIPS])
spread     = spread_bps / 10_000.0

missing_spread = np.isnan(spread).sum()
print(f"Spread coverage : {N - missing_spread}/{N} bonds  ({missing_spread} missing)")
print(f"Spread range    : {spread_bps[~np.isnan(spread_bps)].min():.1f} – "
      f"{spread_bps[~np.isnan(spread_bps)].max():.1f} bps")

c:\Users\lalot\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Spread coverage : 285/303 bonds  (18 missing)
Spread range    : 0.5 – 204.8 bps


---
## 4 — Cashflow Matrix from `Asset_Cashflows`

Builds two aligned matrices:
- `bond_cf` **(T × N)** — daily cashflow grid used to compute duration
- `qtr_bond_cf` **(Q × N)** — quarterly cashflow grid used in the CF shortfall constraint

Cashflows in `Asset_Cashflows` are expressed per **100 face value**.

In [6]:
sql_cf = f"""
SELECT PaymentDate, CUSIP, Payment, Type
FROM `{PROJECT_ID}.{DATASET}.Asset_Cashflows`
WHERE PaymentDate > '{optimization_date.date()}'
  AND CUSIP IN UNNEST({CUSIPS})
"""

cf_raw = client.query(sql_cf).to_dataframe()
cf_raw["PaymentDate"] = pd.to_datetime(cf_raw["PaymentDate"])

print(f"Cashflow rows loaded : {len(cf_raw):,}")
print(f"Date range           : {cf_raw['PaymentDate'].min().date()} → {cf_raw['PaymentDate'].max().date()}")
cf_raw.head()

Cashflow rows loaded : 2,964
Date range           : 2024-04-01 → 2033-02-15


,PaymentDate,CUSIP,Payment,Type
0,2029-10-16,244199BD6,100.0000,PRINCIPAL
1,2029-10-16,244199BD6,2.6875,COUPON
2,2024-07-08,501955AD0,1.1875,COUPON
3,2024-07-08,18977X2C1,1.3250,COUPON
4,2024-07-08,24422EUB3,1.5250,COUPON


In [7]:
# ── Daily cashflow matrix (T × N) ─────────────────────────────────────────
# Sum COUPON + PRINCIPAL on same date for the same CUSIP
cf_agg = (
    cf_raw
    .groupby(["PaymentDate", "CUSIP"])["Payment"]
    .sum()
    .reset_index()
)

cf_pivot = cf_agg.pivot(index="PaymentDate", columns="CUSIP", values="Payment").fillna(0.0)

# Reindex columns to match CUSIPS order; add zeros for CUSIPs with no CF data
cf_pivot = cf_pivot.reindex(columns=CUSIPS, fill_value=0.0)

bond_cf = cf_pivot.values                                          # shape (T, N)
t_dates = cf_pivot.index                                           # DatetimeIndex length T
t_vec   = (t_dates - optimization_date).days.values / 365.25      # years from today
T       = len(t_dates)

print(f"bond_cf shape : {bond_cf.shape}  (T={T} payment dates × N={N} bonds)")

bond_cf shape : (993, 303)  (T=993 payment dates × N=303 bonds)


In [8]:
# ── Quarterly cashflow matrix (Q × N) ─────────────────────────────────────
cf_agg["Quarter"] = cf_agg["PaymentDate"].dt.to_period("Q")

qtr_pivot = (
    cf_agg
    .groupby(["Quarter", "CUSIP"])["Payment"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(columns=CUSIPS, fill_value=0.0)
)

qtr_bond_cf = qtr_pivot.values     # shape (Q, N)
qtr_idx     = qtr_pivot.index      # PeriodIndex length Q
Q           = len(qtr_idx)
fabn_q      = int(qtr_idx.get_loc(pd.Period(FABN_MATURITY, freq="Q")) + 1)  # quarters through FABN maturity (reinvestment horizon)

print(f"qtr_bond_cf shape : {qtr_bond_cf.shape}  (Q={Q} quarters × N={N} bonds)")
# Convert from "per $100 face" to "per $1 face" so that h[i] * CF gives dollars
bond_cf     = bond_cf     / 100.0
qtr_bond_cf = qtr_bond_cf / 100.0
print(f"Quarter range     : {qtr_idx[0]} → {qtr_idx[-1]}")

qtr_bond_cf shape : (36, 303)  (Q=36 quarters × N=303 bonds)
Quarter range     : 2024Q2 → 2033Q1


---
## 5 — Duration

Compute **Macaulay duration** using each bond's own yield as the discount rate:

$$y_i = rf(T_i) + spread_i$$

where $rf(T_i)$ is the **interpolated Treasury rate** at bond $i$'s maturity tenor.
This is the standard approach — `r_FABN` is the liability funding rate and must not be used here.

Fallback to Bloomberg `Mac Dur _Ask_` for bonds with no cashflow data.

In [9]:
import pandas_datareader.data as web
from scipy.interpolate import interp1d
import time

# ── Pull Treasury curve on optimization_date (±7 day window) ──────────────
MATURITIES_YRS = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
FRED_TICKERS   = ["DGS1MO","DGS3MO","DGS6MO","DGS1","DGS2",
                   "DGS3","DGS5","DGS7","DGS10","DGS20","DGS30"]

# Static fallback curve (%), used ONLY if FRED is unreachable (network timeout).
# Source: U.S. Treasury par yields near 2025-01-15.
FRED_FALLBACK = [4.40, 4.35, 4.26, 4.19, 4.27, 4.34, 4.45, 4.55, 4.66, 4.95, 4.88]

def _fetch_treasury_curve(n_retries=3):
    """Fetch the Treasury curve from FRED with retries; fall back to a static curve.

    FRED occasionally times out; a transient failure should not halt the pipeline.
    """
    last_err = None
    for attempt in range(1, n_retries + 1):
        try:
            raw = web.DataReader(
                FRED_TICKERS, "fred",
                start=optimization_date - pd.Timedelta(days=7),
                end=optimization_date + pd.Timedelta(days=7),
            )
            raw.columns = MATURITIES_YRS
            raw = raw.dropna(how="all")
            if raw.empty:
                raise ValueError("FRED returned no rows in the date window")
            day_diff = np.abs((raw.index - optimization_date).days)
            row = raw.iloc[day_diff.argmin()]
            print(f"Treasury curve date used : {row.name.date()}  (FRED, attempt {attempt})")
            return row
        except Exception as e:                       # network / parse failures
            last_err = e
            print(f"  FRED fetch attempt {attempt}/{n_retries} failed: {type(e).__name__}")
            if attempt < n_retries:
                time.sleep(2 * attempt)              # simple backoff: 2s, 4s
    print(f"  WARNING: FRED unreachable ({type(last_err).__name__}). "
          f"Using STATIC fallback Treasury curve (~2025-01-15).")
    return pd.Series(FRED_FALLBACK, index=MATURITIES_YRS, name=optimization_date)

rf_row = _fetch_treasury_curve()
print(rf_row.rename(lambda t: f"{t:.3f}yr").to_string())

# ── Build interpolator: tenor (years) → risk-free rate (decimal) ──────────
valid = rf_row.dropna()
rf_interp = interp1d(
    valid.index.astype(float),
    valid.values / 100.0,      # FRED reports in %, convert to decimal
    kind="linear",
    fill_value="extrapolate",
)

# ── Bond maturity tenor for each CUSIP ────────────────────────────────────
mat_years = ((fixed["maturity"] - optimization_date).dt.days / 365.25).values
mat_years = np.clip(mat_years, MATURITIES_YRS[0], MATURITIES_YRS[-1])

rf_per_bond    = rf_interp(mat_years)        # shape (N,) — risk-free rate per bond
yield_per_bond = rf_per_bond + spread        # y_i = rf(T_i) + spread_i

# ── Modified duration via shared fabn_finance (Macaulay / (1+y)) ──────────
# Discount each bond at its own yield (rf + spread); where spread is NaN, fall
# back to the risk-free rate. Bonds with no cashflow data fall back to BBG.
disc_yield = np.where(np.isnan(yield_per_bond), rf_per_bond, yield_per_bond)
bbg_dur    = fixed["mac_dur_bbg"].values
mod_dur_calc = ff.modified_durations(bond_cf, t_vec, disc_yield)   # NaN where uncomputable
durs         = np.where(np.isnan(mod_dur_calc), bbg_dur, mod_dur_calc)

n_computed = int((~np.isnan(mod_dur_calc)).sum())
n_fallback = int(np.isnan(mod_dur_calc).sum())
print(f"\nDuration computed from cashflows : {n_computed}")
print(f"Duration from BBG fallback       : {n_fallback}")
print(f"Duration range                   : {np.nanmin(durs):.2f} – {np.nanmax(durs):.2f} yrs")
print(f"Mean bond yield used             : {np.nanmean(yield_per_bond)*100:.3f}%")

Treasury curve date used : 2024-03-28  (FRED, attempt 1)
0.083yr     5.49
0.250yr     5.46
0.500yr     5.38
1.000yr     5.03
2.000yr     4.59
3.000yr     4.40
5.000yr     4.21
7.000yr     4.20
10.000yr    4.20
20.000yr    4.45
30.000yr    4.34

Duration computed from cashflows : 303
Duration from BBG fallback       : 0
Duration range                   : 1.84 – 6.90 yrs
Mean bond yield used             : 5.081%


---
## 6 — C1 Capital Factor (theta)

Match each bond's S&P composite rating → NAIC C-1 charge factor `θ_i`.

Fallback chain: `BBG Composite (S&P)` → `Moody's` → default IG factor.

In [10]:
# ── C1 RBC charge factor θ_i (NAIC rating → factor) ───────────────────────
# Table + lookup live in fabn_finance (shared, unit-tested). Fallback chain:
# S&P (BBG Composite) → Moody's → BBB default. `fixed` is already in CUSIPS
# order, so its rating columns align with the bond index directly (no per-bond
# DataFrame scan — was O(N²), now a single vectorised pass).
theta = ff.c1_factors(fixed["rating_sp"].values, fixed["rating_moodys"].values)

n_sp = int(fixed["rating_sp"].apply(lambda r: ff._clean_rating(r) in ff.C1_SP).sum())
n_default = int(sum(
    ff._clean_rating(s) not in ff.C1_SP and ff._clean_rating(m) not in ff.C1_MOODYS
    for s, m in zip(fixed["rating_sp"].values, fixed["rating_moodys"].values)
))

print(f"theta range : {theta.min():.5f} – {theta.max():.5f}")
print(f"Mean C1     : {theta.mean():.5f}  ({theta.mean()*100:.3f}%)")
print(f"Rating source : {n_sp} S&P, {theta.size - n_sp - n_default} Moody's fallback, "
      f"{n_default} BBB default")

theta range : 0.00158 – 0.02168
Mean C1     : 0.00911  (0.911%)
Rating source : 303 S&P, 0 Moody's fallback, 0 BBB default


---
## 7 — Current Allocations, Signal, Transaction Costs

- `h_curr` : equal-weight for now (placeholder until real portfolio is loaded)
- `tau`    : set to 0 (deferred)
- `signal` : set to 0 (deferred)

In [11]:
# Equal-weight current allocation
h_curr = np.full(N, H / N)

# Deferred — set to zero until ready
tau    = np.zeros(N)   # transaction cost per bond
signal = np.zeros(N)   # composite signal per bond

print(f"h_curr (equal-weight) : ${H/N:,.2f} per bond")

h_curr (equal-weight) : $1,650,165.02 per bond


---
## 8 — FABN Liability Cashflow Schedule & D_FABN

Build the exact semi-annual payment schedule from the FABN terms:
- **Issue**: 2022-09-06 | **Maturity**: 2027-09-06
- **Coupon**: 3.205% annual, paid semi-annually → 1.6025% per period
- **Principal**: repaid in full at maturity

Only future payments (after `optimization_date`) enter the optimizer.

In [12]:
# ── Build full semi-annual payment schedule ───────────────────────────────
semi_coupon = FABN_COUPON / 2          # 1.6025% per period
face        = 100.0                    # per 100 face value

# Semi-annual coupon dates on the Sep-6 / Mar-6 anniversary, from the first
# coupon (issue + 6mo) through maturity. Number of periods derived directly
# from the issue→maturity tenor (≈10 for the 5y FABN) — no intermediate
# date_range needed.
n_periods  = round((FABN_MATURITY - FABN_ISSUE).days / 365.25 * 2)
fabn_dates = pd.DatetimeIndex(
    [FABN_ISSUE + pd.DateOffset(months=6 * k) for k in range(1, n_periods + 1)]
)

fabn_cf_full = pd.DataFrame({
    "date":      fabn_dates,
    "coupon":    semi_coupon * face,
    "principal": [0.0] * (len(fabn_dates) - 1) + [face],
})
fabn_cf_full["total"] = fabn_cf_full["coupon"] + fabn_cf_full["principal"]

print("Full FABN schedule (per 100 face):")
print(fabn_cf_full.to_string(index=False))

# ── Keep only future payments (after optimization_date) ───────────────────
fabn_future = fabn_cf_full[fabn_cf_full["date"] > optimization_date].copy()
fabn_future["t_years"] = (fabn_future["date"] - optimization_date).dt.days / 365.25

print(f"\nFuture payments from {optimization_date.date()}:")
print(fabn_future[["date", "coupon", "principal", "total", "t_years"]].to_string(index=False))

# ── Compute D_FABN (Macaulay → Modified duration) ─────────────────────────
total_pv   = fabn_future["total"].sum()
mac_D_FABN = (fabn_future["t_years"] * fabn_future["total"]).sum() / total_pv
D_FABN     = mac_D_FABN / (1 + r_FABN / 2)   # semi-annual compounding convention

print(f"\nMacaulay D_FABN : {mac_D_FABN:.4f} yrs")
print(f"Modified D_FABN : {D_FABN:.4f} yrs  ← used in optimizer")

# ── Map FABN cashflows to the quarterly grid (aligned with qtr_bond_cf) ───
fabn_future["quarter"] = fabn_future["date"].dt.to_period("Q")

# Build Series indexed by the same quarter labels as qtr_idx
fabn_qtr_series = fabn_future.groupby("quarter")["total"].sum()

# Scale from per-100 to actual dollars using H
# (bond CFs are also per-100 and scaled inside the optimizer via h_i)
qtr_fabn_cf = np.array([
    fabn_qtr_series.get(q, 0.0) * (H / face)
    for q in qtr_idx
])

print(f"\nqtr_fabn_cf (${H/1e6:.0f}M face, non-zero quarters):")
for q, v in zip(qtr_idx, qtr_fabn_cf):
    if v > 0:
        print(f"  {q}  ${v:>14,.2f}")

# ── Score ──────────────────────────────────────────────────────────────────
score = spread + beta_w * signal

print(f"\nscore range : {score[~np.isnan(score)].min()*10000:.1f} – "
      f"{score[~np.isnan(score)].max()*10000:.1f} bps")

Full FABN schedule (per 100 face):
      date  coupon  principal    total
2023-03-06  1.6025        0.0   1.6025
2023-09-06  1.6025        0.0   1.6025
2024-03-06  1.6025        0.0   1.6025
2024-09-06  1.6025        0.0   1.6025
2025-03-06  1.6025        0.0   1.6025
2025-09-06  1.6025        0.0   1.6025
2026-03-06  1.6025        0.0   1.6025
2026-09-06  1.6025        0.0   1.6025
2027-03-06  1.6025        0.0   1.6025
2027-09-06  1.6025      100.0 101.6025

Future payments from 2024-03-29:
      date  coupon  principal    total  t_years
2024-09-06  1.6025        0.0   1.6025 0.440794
2025-03-06  1.6025        0.0   1.6025 0.936345
2025-09-06  1.6025        0.0   1.6025 1.440110
2026-03-06  1.6025        0.0   1.6025 1.935661
2026-09-06  1.6025        0.0   1.6025 2.439425
2027-03-06  1.6025        0.0   1.6025 2.934976
2027-09-06  1.6025      100.0 101.6025 3.438741

Macaulay D_FABN : 3.2874 yrs
Modified D_FABN : 3.2355 yrs  ← used in optimizer

qtr_fabn_cf ($500M face, non-zero qua

---
## 9 — Validation

Check shapes, NaN coverage, and key alignment before handing off to the optimizer.

In [13]:
checks = [
    ("bond_cf shape",      bond_cf.shape == (T, N)),
    ("qtr_bond_cf shape",  qtr_bond_cf.shape == (Q, N)),
    ("durs length",        len(durs) == N),
    ("theta length",       len(theta) == N),
    ("spread length",      len(spread) == N),
    ("h_curr length",      len(h_curr) == N),
    ("score length",       len(score) == N),
    ("qtr_fabn_cf length", len(qtr_fabn_cf) == Q),
    ("no NaN in durs",     not np.isnan(durs).any()),
    ("no NaN in theta",    not np.isnan(theta).any()),
    ("spread coverage",    (~np.isnan(spread)).mean() > 0.8),
]

all_pass = True
for name, result in checks:
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  [{status}]  {name}")

print()
if all_pass:
    print("All checks passed — pipeline ready.")
else:
    print("Some checks FAILED — review before running the optimizer.")

  [PASS]  bond_cf shape
  [PASS]  qtr_bond_cf shape
  [PASS]  durs length
  [PASS]  theta length
  [PASS]  spread length
  [PASS]  h_curr length
  [PASS]  score length
  [PASS]  qtr_fabn_cf length
  [PASS]  no NaN in durs
  [PASS]  no NaN in theta
  [PASS]  spread coverage

All checks passed — pipeline ready.


In [14]:
# Fill any remaining NaN spreads with sector median before passing to optimizer
sector_map = fixed.set_index("CUSIP")["sector"]
spread_series = pd.Series(spread, index=CUSIPS)
sector_medians = spread_series.groupby(sector_map).transform("median")
spread_clean = spread_series.fillna(sector_medians).fillna(spread_series.median()).values
score_clean  = spread_clean + beta_w * signal

n_filled = np.isnan(spread).sum()
print(f"NaN spreads filled with sector median : {n_filled}")

NaN spreads filled with sector median : 18


---
## 9.5 — Prices, Book Yield, Amortization & Bid-Ask Transaction Costs

Statutory-accounting inputs derived from the daily price tables
(`Mid_Price`, `Bid_Price`, `Ask_Price`). Under SAP, bonds are held at **amortized
cost**, so earnings come from **book yield** (coupon + amortization), not market
appreciation. This section produces:

| Key | Description |
|---|---|
| `price` | mid price per 100 face at the optimization date — the book / purchase value $P_i$ |
| `book_yield` | effective-interest yield $y_i$: the IRR solving $\sum_t CF_{i,t}(1+y_i)^{-t} = P_i$ |
| `coupon_inc` | current coupon yield $= \text{annual coupon} / P_i$ |
| `amort_inc` | amortization/accretion yield $= y_i - \text{coupon\_inc}_i$ |
| `tau` | transaction cost $= (ask_i - bid_i) / (2\,mid_i)$ — relative half bid-ask spread |

Together, $NII_i$ per dollar invested $= y_i = \text{coupon\_inc}_i + \text{amort\_inc}_i$,
which is exactly the statutory $C_{i,t} + A_{i,t}$ from the SAP document.

In [15]:
# =============================================================================
# 9.5 — Prices, Book Yield, Amortization & Bid-Ask Transaction Costs
# =============================================================================
PRICE_DATASET = {
    "mid": "insurance-backed-securities.Mid_Price.mid_long_raw",
    "bid": "insurance-backed-securities.Bid_Price.bid_long_raw",
    "ask": "insurance-backed-securities.Ask_Price.ask_long_raw",  # Price stored as STRING
}

def _nearest_price_map(table, cast_float=False):
    """Price per CUSIP on (or nearest to) optimization_date, +/- 10 day window."""
    price_expr = "SAFE_CAST(Price AS FLOAT64)" if cast_float else "Price"
    sql = f"""
    WITH ranked AS (
        SELECT CUSIP, {price_expr} AS Price, Date,
               ROW_NUMBER() OVER (
                   PARTITION BY CUSIP
                   ORDER BY ABS(DATE_DIFF(Date, DATE '{optimization_date.date()}', DAY))
               ) AS rn
        FROM `{table}`
        WHERE Date BETWEEN DATE_SUB(DATE '{optimization_date.date()}', INTERVAL 10 DAY)
                       AND DATE_ADD(DATE '{optimization_date.date()}', INTERVAL 10 DAY)
    )
    SELECT CUSIP, Price FROM ranked WHERE rn = 1
    """
    return client.query(sql).to_dataframe().set_index("CUSIP")["Price"]

mid_map = _nearest_price_map(PRICE_DATASET["mid"])
bid_map = _nearest_price_map(PRICE_DATASET["bid"])
ask_map = _nearest_price_map(PRICE_DATASET["ask"], cast_float=True)   # cast STRING -> float

# Align to CUSIP order (per 100 face)
mid_raw = np.array([mid_map.get(c, np.nan) for c in CUSIPS])
bid     = np.array([bid_map.get(c, np.nan) for c in CUSIPS])
ask     = np.array([ask_map.get(c, np.nan) for c in CUSIPS])

# Mid price = book / purchase value. Fallback to par (100) where no quote exists.
price = np.where(np.isnan(mid_raw) | (mid_raw <= 0), 100.0, mid_raw)

# ── Book yield via effective-interest IRR (shared fabn_finance) ───────────
# PV(bond_cf_i @ y) = price_i / 100; bond_cf is per $1 face. Where the solve
# fails, fall back to the rf + spread yield used for duration.
annual_coupon = fixed.set_index("CUSIP").loc[CUSIPS, "coupon"].values / 100.0   # per $1 face
fallback_y    = np.nan_to_num(rf_per_bond) + spread_clean

book_yield_raw = ff.book_yields(bond_cf, t_vec, price)              # NaN where no IRR root
n_irr_fail     = int(np.isnan(book_yield_raw).sum())
book_yield     = np.where(np.isnan(book_yield_raw), fallback_y, book_yield_raw)

# ── Coupon / amortization decomposition (statutory C_i + A_i, per $ invested) ──
coupon_inc, amort_inc = ff.coupon_amort_split(book_yield, annual_coupon, price)

# ── Bid-ask transaction cost: relative half-spread per $ traded ───────────────
with np.errstate(invalid="ignore", divide="ignore"):
    tau = (ask - bid) / (2.0 * price)
tau_valid = np.isfinite(tau) & (tau > 0)
n_tau_fill = int((~tau_valid).sum())
tau = np.where(tau_valid, tau, np.nan)
tau = np.where(np.isnan(tau), np.nanmedian(tau), tau)   # median fill for missing quotes

n_mid = int((~np.isnan(mid_raw)).sum())
print(f"Mid price coverage : {n_mid}/{N} bonds  ({N - n_mid} filled at par)")
print(f"Book yield IRR     : {N - n_irr_fail}/{N} solved  ({n_irr_fail} fell back to rf+spread)")
print(f"Bid-ask tau        : {N - n_tau_fill}/{N} from quotes  ({n_tau_fill} median-filled)")
print(f"Book yield         : {np.nanmin(book_yield)*100:.2f}% – {np.nanmax(book_yield)*100:.2f}%  "
      f"(mean {np.nanmean(book_yield)*100:.2f}%)")
print(f"Coupon yield mean  : {np.nanmean(coupon_inc)*100:.2f}%   "
      f"Amort yield mean : {np.nanmean(amort_inc)*100:+.3f}%")
print(f"Bid-ask tau        : {np.nanmin(tau)*1e4:.1f} – {np.nanmax(tau)*1e4:.1f} bps  "
      f"(mean {np.nanmean(tau)*1e4:.1f} bps)")

c:\Users\lalot\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Mid price coverage : 286/303 bonds  (17 filled at par)
Book yield IRR     : 303/303 solved  (0 fell back to rf+spread)
Bid-ask tau        : 284/303 from quotes  (19 median-filled)
Book yield         : 4.42% – 7.12%  (mean 5.41%)
Coupon yield mean  : 4.30%   Amort yield mean : +1.116%
Bid-ask tau        : 0.1 – 65.1 bps  (mean 13.9 bps)


---
## 10 — Pipeline Output

All optimizer inputs collected in a single `pipeline` dict.  
Import or `%run` this notebook from the optimizer notebook to access them.

In [ ]:
# =============================================================================
# CVaR SCENARIOS (Step 4) -- historical rate+spread shocks + per-bond loss coeffs
# =============================================================================
# Robin's tail risk = the book-value-vs-market-value gap at a forced unwind. We
# build the loss distribution from HISTORICAL joint moves of a benchmark Treasury
# rate (DGS5) and the IG corporate OAS (BAMLC0A0CM) -- real data, no distributional
# assumption -- then reprice every bond's cashflows under each shock.
# relloss[s,i] = 1 - MV_i(shock_s)/BV_i is the per-$ forced-sale loss (linear in the
# holdings) that feeds a Rockafellar-Uryasev CVaR limit in the optimizer.
CVAR_HORIZON_DAYS = 21     # ~1 trading month change horizon
CVAR_MAX_SCEN     = 250
CVAR_ALPHA        = 0.95   # tail level (worst 5%)

def _fetch_shock_history(n_retries=3):
    """~2yr history of 5yr UST (DGS5) and IG corp OAS (BAMLC0A0CM) from FRED."""
    for attempt in range(1, n_retries + 1):
        try:
            raw = web.DataReader(["DGS5", "BAMLC0A0CM"], "fred",
                                 start=optimization_date - pd.Timedelta(days=760),
                                 end=optimization_date).dropna(how="any")
            if len(raw) < CVAR_HORIZON_DAYS + 10:
                raise ValueError("insufficient FRED history")
            print(f"CVaR shock history: {len(raw)} days DGS5+IG-OAS (FRED, attempt {attempt})")
            return raw["DGS5"].values / 100.0, raw["BAMLC0A0CM"].values / 100.0
        except Exception as e:
            print(f"  CVaR-history FRED attempt {attempt}/{n_retries} failed: {type(e).__name__}")
            if attempt < n_retries: time.sleep(2 * attempt)
    print("  WARNING: FRED history unreachable -- using parametric fallback shocks.")
    rng = np.random.default_rng(0); n = 250
    dr = rng.normal(0.0, 0.0035, n)             # ~35bp monthly rate vol
    ds = 0.3 * dr + rng.normal(0.0, 0.0020, n)  # spread partly co-moves with rates
    return None, (dr, ds)

_hist = _fetch_shock_history()
if _hist[0] is None:                            # fallback: raw shocks already
    SCEN_D_RATE, SCEN_D_SPREAD = _hist[1]
else:
    _rate_hist, _spread_hist = _hist
    SCEN_D_RATE, SCEN_D_SPREAD = ff.historical_shock_scenarios(
        _rate_hist, _spread_hist, horizon_days=CVAR_HORIZON_DAYS, max_scenarios=CVAR_MAX_SCEN)

# Per-bond forced-sale loss coefficients (relative, per $1 allocated) -- linear in h.
# Base yield = book_yield (the IRR that reprices each bond at its current price), so
# BV = zero-shock MV reproduces today's book value.
_bv0     = ff.market_values_under_shocks(bond_cf, t_vec, book_yield,
                                         np.array([0.0]), np.array([0.0]))[0]    # (N,)
_mv_scen = ff.market_values_under_shocks(bond_cf, t_vec, book_yield,
                                         SCEN_D_RATE, SCEN_D_SPREAD)             # (S,N)
CVAR_RELLOSS = 1.0 - _mv_scen / np.where(_bv0 > 1e-9, _bv0, 1.0)                # (S,N)
_pavg = np.sort(CVAR_RELLOSS.mean(axis=1))
print(f"CVaR scenarios: S={len(SCEN_D_RATE)} | rate shock std {SCEN_D_RATE.std()*1e4:.0f}bp"
      f" | spread shock std {SCEN_D_SPREAD.std()*1e4:.0f}bp")
print(f"Portfolio-avg loss: worst-5% mean ~ {_pavg[-max(1,len(_pavg)//20):].mean()*100:+.2f}% of book"
      f" | median {np.median(_pavg)*100:+.2f}%")


In [16]:
pipeline = {
    # ── dimensions ──────────────────────────────────────────────────────────
    "N":              N,
    "T":              T,
    "Q":              Q,
    "fabn_q":         fabn_q,         # quarters through FABN maturity (reinvestment horizon)
    "CUSIPS":         CUSIPS,
    "fixed":          fixed,           # full DataFrame for inspection

    # ── per-bond arrays (shape N) ────────────────────────────────────────────
    "spread":         spread_clean,    # OAS spread in decimal
    "durs":           durs,            # modified duration (years)
    "theta":          theta,           # C-1 RBC charge factor  f_i
    "tau":            tau,             # transaction cost = bid-ask half-spread (decimal)
    "signal":         signal,          # composite signal (deferred → 0)
    "score":          score_clean,     # spread + β*signal
    "h_curr":         h_curr,          # current allocation

    # ── SAP statutory-accounting arrays (shape N) — from Section 9.5 ──────────
    "price":          price,           # mid price per 100 face (book value P_i)
    "book_yield":     book_yield,      # effective-interest yield y_i = coupon_inc + amort_inc
    "coupon_inc":     coupon_inc,      # current coupon yield (statutory C_i, per $)
    "amort_inc":      amort_inc,       # amortization/accretion yield (statutory A_i, per $)

    # ── cashflow matrices ────────────────────────────────────────────────────
    "bond_cf":        bond_cf,         # (T, N)
    "qtr_bond_cf":    qtr_bond_cf,     # (Q, N)
    "qtr_fabn_cf":    qtr_fabn_cf,     # (Q,)  — placeholder, replace with Athene data
    "qtr_idx":        qtr_idx,
    "t_vec":          t_vec,           # (T,) years from optimization_date

    # ── scalar params ────────────────────────────────────────────────────────
    "H":              H,
    "r_FABN":         r_FABN,
    "D_FABN":         D_FABN,
    "C_curr":         C_curr,
    "C_min":          C_min,
    "RBC_bar":        RBC_bar,
    "dt":             dt,
    "gamma_w":        gamma_w,
    "beta_w":         beta_w,
    "alpha_w":        alpha_w,
    "lambda_w":       lambda_w,
    "eps_D":          eps_D,
    # -- CVaR (Step 4) --------------------------------------------------------
    "cvar_d_rate":    SCEN_D_RATE,     # (S,) per-scenario rate shock (swap MV in CVaR)
    "cvar_relloss":   CVAR_RELLOSS,    # (S,N) per-$ forced-sale loss coefficients
    "cvar_alpha":     CVAR_ALPHA,      # CVaR tail level (worst 5%)
}

# Quick summary table
summary = pd.DataFrame([
    ["Universe size (N)",          N,            ""],
    ["Payment dates (T)",          T,            ""],
    ["Quarterly periods (Q)",      Q,            ""],
    ["Spread mean (bps)",          f"{spread_clean.mean()*10000:.1f}", ""],
    ["Book yield mean (%)",        f"{np.nanmean(book_yield)*100:.2f}", ""],
    ["Bid-ask tau mean (bps)",     f"{np.nanmean(tau)*1e4:.1f}",      ""],
    ["Duration mean (yrs)",        f"{durs.mean():.2f}",              ""],
    ["C1 charge mean (%)",         f"{theta.mean()*100:.3f}",         ""],
    ["FABN D target (yrs)",        D_FABN,       ""],
    ["Budget H ($M)",              H/1e6,        ""],
], columns=["Metric", "Value", "Notes"])

display(summary)
print("\npipeline dict ready.")

,Metric,Value,Notes
0,Universe size (N),303,
1,Payment dates (T),993,
2,Quarterly periods (Q),36,
3,Spread mean (bps),74.5,
4,Book yield mean (%),5.41,
5,Bid-ask tau mean (bps),13.9,
6,Duration mean (yrs),3.63,
7,C1 charge mean (%),0.911,
8,FABN D target (yrs),3.235526,
9,Budget H ($M),500.0,



pipeline dict ready.
